# QuAic QNN Color Estimation Training

Geisterボードゲーム用のQNN（量子ニューラルネットワーク）色推定モデルを学習します。

## ワークフロー
1. **QuAic HNN Composer** -> QNNアーキテクチャを設計し、`config.json`をエクスポート
2. **このノートブック** -> モデルを学習し、`weights.pth`をダウンロード
3. **QuAic Competition** -> `weights.pth`をアップロードして競技に参加

---

## 前提条件
- QuAic HNN Composerからエクスポートした`config.json`ファイル
- Googleアカウント（Colab GPUアクセス用）

**GPU推奨**: `Runtime > Change runtime type > T4 GPU`でGPUを有効にしてください

## 1. 環境セットアップ

必要なパッケージをインストールし、学習リポジトリをクローンします。

In [ ]:
# 依存関係をインストール
!pip install -q torch pennylane numpy matplotlib tqdm

# Qugeister学習リポジトリをクローン
import os
import shutil

# 既存のクローンを削除して最新版を取得
if os.path.exists('Qugeister'):
    shutil.rmtree('Qugeister')

!git clone --depth 1 https://github.com/ukinsama/Qugeister.git

# パスを追加
import sys
sys.path.insert(0, 'Qugeister/src')

print("環境セットアップ完了!")

In [ ]:
# GPU確認
import torch

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA Version: {torch.version.cuda}")
    device = 'cuda'
else:
    print("WARNING: GPUが利用できません。CPUでの学習は遅くなります。")
    print("Runtime > Change runtime type > T4 GPUで有効にしてください")
    device = 'cpu'

## 2. config.jsonをアップロード

QuAic HNN Composerからエクスポートした`config.json`をアップロードしてください。

In [ ]:
from google.colab import files
import json

print("config.jsonファイルをアップロードしてください:")
uploaded = files.upload()

# アップロードされた設定ファイルを検索
config_filename = None
for filename in uploaded.keys():
    if filename.endswith('.json'):
        config_filename = filename
        break

if config_filename:
    # 設定を解析して表示
    with open(config_filename, 'r') as f:
        config = json.load(f)

    print(f"\n設定を読み込みました: {config_filename}")
    
    # Qugeisterのローダーを使用
    from qugeister.models.hnn_config_loader import load_hnn_config
    hnn_config = load_hnn_config(config_filename)
    
    print(f"\n=== HNN設定 ===")
    print(f"  量子ビット数: {hnn_config.quantum.n_qubits}")
    print(f"  量子レイヤー数: {hnn_config.quantum.n_layers}")
    print(f"  埋め込み: {hnn_config.quantum.embedding}")
    print(f"  Ansatz: {hnn_config.quantum.ansatz}")
else:
    print("ERROR: JSONファイルがアップロードされていません。config.jsonをアップロードしてください")

## 3. 学習データをダウンロード

学習用の棋譜データをダウンロードします。

### データセット一覧
| データセット | 試合数 | 説明 |
|-------------|--------|------|
| `diverse_agents_3000` | 3000 | 5種類のAIから収集（推奨） |
| `alphazero_mcts100_1000` | 1000 | AlphaZero高品質データ |

### データの形式
- ゲーム状態（7チャネル×8×8ボード表現）
- 敵駒の真の色（教師あり学習用ラベル）

In [ ]:
import os

# データセットURL（GitHub Releaseから自動ダウンロード）
TRAJECTORY_URLS = {
    "alphazero_mcts100_1000": "https://github.com/ukinsama/Qugeister/releases/download/v1.0-data/alphazero_mcts100_1000.pkl",
    # "diverse_agents_3000": "https://github.com/ukinsama/Qugeister/releases/download/v1.0-data/diverse_agents_3000.pkl",
}

# 使用するデータセット（小さい方をデフォルトに）
DATASET_ID = "alphazero_mcts100_1000"
TRAJECTORY_FILE = f"{DATASET_ID}.pkl"

if os.path.exists(TRAJECTORY_FILE):
    print(f"既存のデータを使用: {TRAJECTORY_FILE}")
else:
    url = TRAJECTORY_URLS.get(DATASET_ID)
    if url:
        print(f"データをダウンロード中: {DATASET_ID}")
        print(f"URL: {url}")
        !wget -q --show-progress -O {TRAJECTORY_FILE} {url}
        print(f"ダウンロード完了: {TRAJECTORY_FILE}")
    else:
        print(f"ERROR: {DATASET_ID} のURLが見つかりません")

In [ ]:
# オプション: 手動でデータをアップロードする場合
# （上のセルでダウンロードできない場合のみ使用）

# from google.colab import files
# print("学習データ(.pkl)をアップロードしてください:")
# uploaded_data = files.upload()
# for filename in uploaded_data.keys():
#     if filename.endswith('.pkl'):
#         TRAJECTORY_FILE = filename
#         print(f"データファイル: {TRAJECTORY_FILE}")
#         break

In [ ]:
# データを読み込み
import pickle

with open(TRAJECTORY_FILE, 'rb') as f:
    trajectory_data = pickle.load(f)

print(f"読み込んだ試合数: {len(trajectory_data)}")

## 4. 学習設定

In [ ]:
# 学習パラメータ
EPOCHS = 50           # エポック数（50-100推奨）
BATCH_SIZE = 256      # バッチサイズ（GPU: 256, CPU: 64）
LEARNING_RATE = 0.001 # 学習率
PATIENCE = 10         # Early Stoppingのpatience

# CPU用に調整
if device == 'cpu':
    BATCH_SIZE = 64
    EPOCHS = 20
    print("CPUモード用に設定を調整しました")

print(f"学習設定:")
print(f"  デバイス: {device}")
print(f"  エポック数: {EPOCHS}")
print(f"  バッチサイズ: {BATCH_SIZE}")
print(f"  学習率: {LEARNING_RATE}")

## 5. モデル構築と学習

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm

from qugeister.models.hnn_config_loader import HNNColorEstimator

# モデルを構築（backpropモードで高速化）
model = HNNColorEstimator(hnn_config, device='cpu', backend='backprop')

print(f"\nパラメータ数: {sum(p.numel() for p in model.parameters()):,}")
print(f"\nstate_dictキー:")
for key, value in model.state_dict().items():
    print(f"  {key}: {value.shape}")

In [ ]:
def prepare_data(trajectory_data, train_ratio=0.8):
    """棋譜データから学習用データを準備"""
    X_list = []
    y_list = []

    for traj in trajectory_data:
        # Player A
        if 'states_A' in traj and 'true_colors_A' in traj:
            for state, colors in zip(traj['states_A'], traj['true_colors_A']):
                state = np.array(state).flatten()
                colors = np.array(colors)
                if state.shape[0] == 448 and colors.shape[0] == 8:
                    colors = np.clip(colors, 0, 1)  # -1を0に置換
                    X_list.append(state)
                    y_list.append(colors)

        # Player B
        if 'states_B' in traj and 'true_colors_B' in traj:
            for state, colors in zip(traj['states_B'], traj['true_colors_B']):
                state = np.array(state).flatten()
                colors = np.array(colors)
                if state.shape[0] == 448 and colors.shape[0] == 8:
                    colors = np.clip(colors, 0, 1)
                    X_list.append(state)
                    y_list.append(colors)

    X = np.array(X_list, dtype=np.float32)
    y = np.array(y_list, dtype=np.int64)

    # シャッフル
    indices = np.random.permutation(len(X))
    X, y = X[indices], y[indices]

    # 分割
    split_idx = int(len(X) * train_ratio)
    X_train, X_val = X[:split_idx], X[split_idx:]
    y_train, y_val = y[:split_idx], y[split_idx:]

    print(f"学習データ: {len(X_train):,} サンプル")
    print(f"検証データ: {len(X_val):,} サンプル")

    return X_train, y_train, X_val, y_val

X_train, y_train, X_val, y_val = prepare_data(trajectory_data)

# DataLoader作成
train_dataset = TensorDataset(
    torch.tensor(X_train),
    torch.tensor(y_train)
)
val_dataset = TensorDataset(
    torch.tensor(X_val),
    torch.tensor(y_val)
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

In [ ]:
def train_epoch(model, loader, optimizer, criterion):
    """1エポックの学習"""
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for X_batch, y_batch in loader:
        optimizer.zero_grad()
        outputs = model(X_batch)  # [batch, 8, 2]

        loss = criterion(outputs.view(-1, 2), y_batch.view(-1))
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        preds = outputs.argmax(dim=-1)
        correct += (preds == y_batch).sum().item()
        total += y_batch.numel()

    return total_loss / len(loader), correct / total


def validate(model, loader, criterion):
    """検証"""
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for X_batch, y_batch in loader:
            outputs = model(X_batch)
            loss = criterion(outputs.view(-1, 2), y_batch.view(-1))

            total_loss += loss.item()
            preds = outputs.argmax(dim=-1)
            correct += (preds == y_batch).sum().item()
            total += y_batch.numel()

    return total_loss / len(loader), correct / total

In [ ]:
# オプティマイザと損失関数
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
criterion = nn.CrossEntropyLoss()

# 学習ループ
best_val_acc = 0
patience_counter = 0
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

print("学習を開始します...")
print("注意: 量子層のbackprop微分はCPUで実行されます\n")

for epoch in tqdm(range(EPOCHS), desc='Training'):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion)
    val_loss, val_acc = validate(model, val_loader, criterion)

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        patience_counter = 0
        torch.save(model.state_dict(), 'best_model.pth')
    else:
        patience_counter += 1

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{EPOCHS}")
        print(f"  Train Loss: {train_loss:.4f}, Acc: {train_acc:.4f}")
        print(f"  Val Loss: {val_loss:.4f}, Acc: {val_acc:.4f}")

    # Early Stopping
    if patience_counter >= PATIENCE:
        print(f"\nEarly stopping at epoch {epoch+1}")
        break

print(f"\n学習完了! ベスト検証精度: {best_val_acc:.4f}")

## 6. 学習結果の可視化

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# 損失
ax1.plot(history['train_loss'], label='Train')
ax1.plot(history['val_loss'], label='Validation')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training Loss')
ax1.legend()
ax1.grid(True)

# 精度
ax2.plot([acc * 100 for acc in history['train_acc']], label='Train')
ax2.plot([acc * 100 for acc in history['val_acc']], label='Validation')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.set_title('Training Accuracy')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.savefig('training_history.png', dpi=150)
plt.show()

print(f"\n最終結果:")
print(f"  Train Accuracy: {history['train_acc'][-1]*100:.1f}%")
print(f"  Val Accuracy: {history['val_acc'][-1]*100:.1f}%")

## 7. 学習済み重みをダウンロード

学習済みモデルの重みをダウンロードしてQuAicに提出します。

In [ ]:
from google.colab import files
import os

# ベストモデルを読み込み
model.load_state_dict(torch.load('best_model.pth', weights_only=True))

# エクスポート
model_name = os.path.splitext(config_filename)[0]
weights_path = f'{model_name}_weights.pth'

torch.save(model.state_dict(), weights_path)

size_kb = os.path.getsize(weights_path) / 1024
print(f"学習済み重みを保存: {weights_path} ({size_kb:.1f} KB)")

print(f"\nstate_dictキー:")
for key, value in model.state_dict().items():
    print(f"  {key}: {value.shape}")

In [ ]:
# ダウンロード
print("weights.pthをダウンロード中...")
files.download(weights_path)

print("\n" + "=" * 50)
print("次のステップ:")
print("1. QuAic (https://quaic-api.up.railway.app) にアクセス")
print("2. 「コンペティション > リーダーボード」に移動")
print("3. 「モデルを提出」をクリック")
print("4. ダウンロードした weights.pth ファイルをアップロード")
print("=" * 50)

## トラブルシューティング

### よくある問題

1. **"GPU not available"**
   - `Runtime > Change runtime type > T4 GPU`でGPUを有効にしてください

2. **"No module named 'qugeister'"**
   - セル1を再実行してリポジトリをクローンしてください

3. **"Out of memory"**
   - `BATCH_SIZE`を128または64に減らしてください
   - `Runtime > Restart runtime`でランタイムを再起動

4. **学習が遅い**
   - 量子層はCPUで実行されるため、ある程度の時間がかかります
   - 各エポックは1-5分程度かかることがあります

### ヘルプ
- QuAic Discord: [discord.gg/quaic](https://discord.gg/quaic)
- GitHub Issues: [github.com/ukinsama/Qugeister](https://github.com/ukinsama/Qugeister)